# RAG Pipeline Exercise — Build a Question-Answering System

## What is RAG?

Large Language Models (like GPT-4) are incredibly powerful: you can ask them questions and they generate fluent, convincing answers. But they have a **big limitation**: they can only answer from what they memorized during training. If you ask about something they haven't seen — for example, a company's internal documents, or news from last week — they might **hallucinate**: confidently make up an answer that sounds right but is completely wrong.

**Retrieval-Augmented Generation (RAG)** solves this problem by adding a **search step** before generating the answer.

### A real-world analogy

Imagine you're taking an exam:
- **Without RAG** = a **closed-book exam**: you answer from memory. If you don't remember something, you might guess (and guess wrong).
- **With RAG** = an **open-book exam**: before answering, you search your textbook for the relevant pages, read them, and *then* answer. Much more reliable!

### The RAG Pipeline — Step by Step

```
Documents → Chunk → Embed → Store in Vector DB
                                    ↓
User Query → Embed Query → Retrieve Top-k → Augment Prompt → LLM → Answer
```

Here is what each step does (don't worry, we will go through each one in detail):

| Step | What it does | Analogy |
|------|-------------|---------|
| **Chunk** | Split long documents into shorter passages (~500 chars each) | Cutting a book into individual pages |
| **Embed** | Convert each passage into a list of numbers (a vector) that captures its meaning | Creating a "fingerprint" of the text's meaning |
| **Store** | Save all vectors in a special database optimized for finding similar items | Filing the fingerprints in a cabinet |
| **Retrieve** | When a user asks a question, find the most similar passages | Looking up the most relevant pages |
| **Augment** | Insert those passages into the LLM's prompt | Handing the relevant pages to the student |
| **Answer** | The LLM reads the passages and generates an answer | The student reads and answers |

## How to work through this notebook

Most of the code is already written for you. Look for the 🎯 symbol — it marks the **only** lines you need to modify. The exercises are designed as **fill-in-the-blank**: you just need to replace `___` with the correct value. Each 🎯 line tells you exactly what to write.

There are also **Discussion** questions after each task: answer them in your own words (no coding needed).

**Estimated time:** 2–3 hours.

---
## Setup

Run the cells below to install dependencies, import libraries, and set your API key.

| Library | What it does in our exercise |
|---------|------------------------------|
| `datasets` | Downloads the SQuAD dataset from HuggingFace (a hub for AI datasets) |
| `langchain` | Framework that connects the pieces of our RAG pipeline together, like LEGO blocks |
| `chromadb` | Vector database — a "smart filing cabinet" that finds similar documents |
| `openai` | Lets us use OpenAI's models for embeddings and answers |
| `helper` | Our utility module with plotting and display functions |

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/rag_exercise')

sys.path.insert(0, '.')

!pip install datasets langchain langchain-openai langchain-community chromadb openai tiktoken matplotlib numpy --quiet
print('Setup complete!')

In [ ]:
import helper
from helper import format_docs

from datasets import load_dataset
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
print('RAG Pipeline loaded!')

In [ ]:
# ── API Key ────────────────────────────────────────────────────────────────
# os.environ["OPENAI_API_KEY"] = "sk-..."
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key
except (ImportError, Exception):
    pass

assert os.environ.get("OPENAI_API_KEY"), "Set your OPENAI_API_KEY before continuing."
print('API key set!')

---
## Task 1 — Data Loading & Exploration

### What we're doing and why

Every RAG system needs a **knowledge base** — a collection of documents that the system can search through. Think of it as the "textbook" that our system will consult during the open-book exam.

For this exercise, we use Wikipedia paragraphs from the **SQuAD dataset** (Stanford Question Answering Dataset). SQuAD is a famous dataset used in AI research: it contains thousands of question–answer pairs, where each question refers to a specific Wikipedia paragraph.

**Example from SQuAD:**
> **Paragraph**: "The Normans were the people who gave their name to Normandy, a region in France..."
> **Question**: "In what country is Normandy located?"
> **Answer**: "France"

Since multiple questions can refer to the same paragraph, we first **deduplicate** (remove duplicates from) the paragraphs to get our unique list of documents.

### Nothing to code here — just run and observe!

Pay attention to:
- **How many** unique documents we have
- **What** a typical document looks like (it's a Wikipedia paragraph)
- **How long** the documents are (in characters) — this will matter later!

In [ ]:
# ── Load the SQuAD v2 dataset from HuggingFace ───────────────────────────
squad = load_dataset("rajpurkar/squad_v2", split="validation")

print(f"Total examples in validation set: {len(squad)}")
print(f"Each example contains: {list(squad.features.keys())}")
print(f"We care about 'context' (the Wikipedia paragraph) and 'question'/'answers'.")

In [ ]:
# ── Extract unique documents ──────────────────────────────────────────────
all_documents = list(dict.fromkeys(squad["context"]))
documents = all_documents[:200]

print(f"Unique documents: {len(all_documents)} | Working set: {len(documents)}")
print(f"\nFirst document (first 300 chars):")
print(f"{documents[0][:300]}...")

In [ ]:
# ── Visualize document length distribution ───────────────────────────────
helper.plot_document_length_distribution(documents)

### What just happened?

The histogram above shows how long each document is (measured in characters). You can see:
- The **mean** (average) length — shown as a red dashed line
- The **median** (the "middle" document if you sorted them by length) — shown as a green dashed line

Notice that documents have very different lengths! Some are short (a few hundred characters) and some are quite long (over a thousand characters). This is important because in the next step, we will need to **cut** long documents into smaller pieces.

### Discussion

Look at the histogram and think about these questions:
1. What is the typical length of a document (roughly, in characters)?
2. Are there documents that are much longer or shorter than the average?
3. If we want to feed these documents to an AI model that works best with ~500 characters at a time, what problem do we have?

*Your answer:* [write here]

---
## Task 2 — Document Chunking

### Why do we need to cut documents into pieces?

Imagine you have a 10-page article about World War II. If someone asks "When did D-Day happen?", the answer is in just one paragraph. But if we store the entire article as a single unit, our search system would return *all 10 pages* — and the LLM would have to read through everything to find the one relevant paragraph.

**Chunking** = splitting long documents into shorter, focused passages. This helps in two ways:

1. **Better search precision**: a short chunk about "D-Day, June 6, 1944" will match the question much better than a long article that mentions D-Day once among many other topics.
2. **Fits model limits**: embedding models (which we'll use in the next step) have a maximum input length. Shorter chunks ensure we stay within limits.

### How does `RecursiveCharacterTextSplitter` work?

LangChain provides a smart splitter that doesn't just cut text at random positions. Instead, it tries to split at **natural boundaries**, in this priority order:

1. **Double newlines** (paragraph breaks) — best: keeps whole paragraphs together
2. **Single newlines** — good: keeps sentences together
3. **Spaces** (between words) — okay: at least doesn't cut words in half
4. **Individual characters** — last resort

We configure it with two numbers:
- **`chunk_size=500`** — each chunk should be at most ~500 characters long
- **`chunk_overlap=50`** — the last 50 characters of one chunk are repeated at the start of the next chunk

**Why overlap?** Imagine a sentence that says "Napoleon was born in **Corsica** in 1769". If the chunk boundary falls right between "Corsica" and "in 1769", one chunk would have the place without the date, and the other would have the date without the place. Overlap ensures that boundary information isn't lost.

---

### 🎯 What you need to do

The code in the next cell is **almost complete**. There are two lines marked `# ← COMPLETE THIS LINE`. You need to:

1. **Line 1** — Set the chunk size to `500` and overlap to `50`. The code already has the structure, you just need to type the numbers in place of `___`.
2. **Line 2** — Tell the splitter which variable contains our documents. The variable is called `documents` (we created it in Task 1).

**This is what the completed code should look like:**
```python
# Line 1:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Line 2:
chunks = text_splitter.create_documents(documents)
```

In [ ]:
# ── Task 2: Create the text splitter and split documents ─────────────────

# 🎯 COMPLETE THIS LINE: replace ___ with the numbers 500 and 50
text_splitter = RecursiveCharacterTextSplitter(chunk_size=___, chunk_overlap=___)

# 🎯 COMPLETE THIS LINE: replace ___ with the word: documents
chunks = text_splitter.create_documents(___)

print(f"Split {len(documents)} documents → {len(chunks)} chunks "
      f"(avg {len(chunks)/len(documents):.1f} per doc)")
print(f"\nFirst chunk:\n{chunks[0].page_content[:200]}...")

In [ ]:
# ── Visualize chunk length distribution ──────────────────────────────────
helper.plot_chunk_length_distribution(chunks)

### What just happened?

Look at the chunk length histogram above and compare it with the document length histogram from Task 1. Notice how:
- **Before chunking**: document lengths varied a lot (some very short, some very long)
- **After chunking**: most chunks are close to our target of 500 characters

The splitter cut long documents into multiple chunks and left short documents as single chunks. The overlap means some information appears in two consecutive chunks — this is intentional!

### Discussion

Think about the trade-offs (no coding needed — just think and write):

1. **Very small chunks** (e.g., 50 characters, just a sentence fragment): Would the search system be able to understand what a chunk is about from just a few words? Would a chunk like "in France during the" be useful?

2. **Very large chunks** (e.g., 5000 characters, an entire article): If a chunk covers 10 different topics, and the user asks about one of them, would the embedding (the "fingerprint") of that chunk accurately represent that specific topic?

3. **The overlap** (`chunk_overlap=50`): Can you think of a sentence that might get split in half without overlap? What would go wrong?

*Your answer:* [write here]

---
## Task 3 — Embeddings & Vector Store

### What are embeddings? (The core idea!)

This is one of the most important concepts in modern AI. An **embedding** is a way to represent text as a **list of numbers** (called a "vector"). The magic is that **texts with similar meanings get similar numbers**.

**Example:**
- "The cat sat on the mat" → `[0.12, -0.45, 0.78, ...]` (1536 numbers)
- "A kitten rested on the rug" → `[0.11, -0.43, 0.80, ...]` (very similar numbers!)
- "The stock market crashed" → `[0.89, 0.23, -0.56, ...]` (very different numbers!)

Even though "cat" and "kitten" are different words, the embedding model understands they mean similar things. This is what makes semantic search possible: instead of matching exact words (like Ctrl+F), we match **meanings**.

### What is a vector store?

A **vector store** (we use ChromaDB) is a database specifically designed for storing and searching these number-lists efficiently. It's like a library catalog, but instead of searching by title or author, you search by *meaning*.

When you search, ChromaDB:
1. Converts your question into numbers (embedding)
2. Compares those numbers against all stored chunks
3. Returns the chunks with the most similar numbers

### What happens step by step when we run `Chroma.from_documents()`

1. Each chunk's text is sent to OpenAI's embedding model (via the API)
2. The model converts each chunk into a list of 1536 numbers
3. ChromaDB stores both the original text AND its numbers
4. Later, when you search, your question is also converted to numbers, and ChromaDB finds the closest match

---

### 🎯 What you need to do

The code in the next cell has two lines to complete:

1. **Line 1** — Choose which embedding model to use. We want OpenAI's small and fast model. You just need to type the model name `"text-embedding-3-small"` where you see `___`.
2. **Line 2** — Build the vector store. `Chroma.from_documents()` needs two things: **what** to store (our `chunks`) and **how** to embed them (our `embeddings` model). Type those two variable names where you see `___`.

**This is what the completed code should look like:**
```python
# Line 1:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Line 2:
vectorstore = Chroma.from_documents(chunks, embeddings)
```

In [ ]:
# ── Task 3: Create embedding model and vector store ──────────────────────

# 🎯 COMPLETE THIS LINE: replace ___ with the model name "text-embedding-3-small" (with quotes!)
embeddings = OpenAIEmbeddings(model=___)

# 🎯 COMPLETE THIS LINE: replace the two ___ with: chunks, embeddings
vectorstore = Chroma.from_documents(___, ___)

print(f"Vector store ready — {vectorstore._collection.count()} vectors indexed.")

### Testing retrieval — does our search actually work?

Let's test the vector store by asking it to find relevant chunks for some queries. We use `similarity_search_with_score` which returns both the matching chunks and a **distance score**.

**How to read the scores:**
- ChromaDB uses **distance** (not similarity), so **lower = more similar** (like physical distance: closer = more related)
- A score close to **0** = the query and chunk are very similar in meaning
- A **high score** = they are not related

We test with three queries on purpose:
- Two **in-domain** queries (topics that should be in our Wikipedia knowledge base)
- One **out-of-domain** query (pizza toppings — definitely NOT in Wikipedia's SQuAD subset)

**Pay attention to:** What happens with the pizza question? Does the system know it has no relevant information?

In [ ]:
# ── Test retrieval ────────────────────────────────────────────────────────
test_queries = [
    "Who was the first president of the United States?",
    "What is photosynthesis?",
    "What is the best pizza topping?",  # out-of-domain!
]

for query in test_queries:
    results = vectorstore.similarity_search_with_score(query, k=3)
    helper.display_retrieval_results(query, results)

### What just happened?

Look carefully at the results above. You should notice something important about the **pizza question**: the vector store **still returned results**, even though our knowledge base has nothing about pizza! It returned the "least unrelated" chunks it could find.

This is a fundamental property of vector search: **it always returns something**. It can't say "I have nothing relevant." It just finds the closest matches, even if they're not very close. (Notice the higher distance scores for the pizza query.)

This is why, in the next step, we'll need to tell the LLM: "If the context doesn't help, say 'I don't know'."

### Discussion

1. For the in-domain queries (president, photosynthesis): Do the retrieved passages look relevant? Could you answer the question using just those passages?
2. For the pizza query: What did the vector store return? Is it useful at all?
3. Compare the **scores**: Are the scores for the pizza query higher (= less similar) than for the in-domain queries?
4. **Key insight**: Why is it dangerous that the vector store always returns something, even for irrelevant questions?

*Your answer:* [write here]

---
## Task 4 — Build the RAG Chain

### Connecting all the pieces

Now we connect all the components into a working pipeline. Think of it like an assembly line in a factory:

```
User's question
      ↓
[1. RETRIEVER] → searches the vector store → finds relevant chunks
      ↓
[2. FORMAT]    → joins the chunks into a single text block called "context"
      ↓
[3. PROMPT]    → inserts the context + question into a template for the LLM
      ↓
[4. LLM]       → reads the prompt and generates an answer
      ↓
[5. PARSER]    → extracts just the text from the LLM's response
      ↓
Final answer!
```

LangChain lets us build this assembly line using the **pipe operator `|`**, similar to how Unix pipes work: the output of one step becomes the input of the next.

### What is a retriever?

A **retriever** is just a wrapper around our vector store that makes it compatible with the chain. When called with a question, it:
1. Embeds the question into a vector
2. Searches the vector store for the 4 most similar chunks (`k=4`)
3. Returns those chunks

### What is a prompt template?

The **prompt template** is the set of instructions we give to the LLM. It's like briefing a human assistant before asking them a question. It contains two special **placeholders** that LangChain fills in automatically:
- **`{context}`** → gets replaced with the retrieved chunks
- **`{question}`** → gets replaced with the user's question

A good RAG prompt must tell the LLM two critical rules:
1. **Answer ONLY from the context** — otherwise the LLM ignores the retrieved documents and answers from memory (defeating the purpose of RAG!)
2. **Say "I don't know" if the context doesn't help** — remember the pizza question? The vector store always returns *something*, even for irrelevant queries. Without this rule, the LLM would make up answers.

---

### 🎯 What you need to do

In the next cell, you'll see a prompt template with a gap. You need to **write 2–3 sentences of instructions** in plain English. There's no single correct answer — write it in your own words!

Here's an example of what you could write (but try your own version!):
```
You are a helpful assistant. Answer the question using ONLY the context provided below.
If the context does not contain the answer, say "I don't know".
Keep your answer short.
```

**Important:** Don't touch the `{context}` and `{question}` parts — they must stay exactly as they are.

In [ ]:
# ── Create retriever and LLM ──────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print('Retriever (k=4) and LLM (gpt-4o-mini) ready!')

In [ ]:
# ── Task 4: Write the prompt template ────────────────────────────────────
#
# 🎯 WRITE YOUR INSTRUCTIONS: replace the line "___WRITE YOUR INSTRUCTIONS HERE___"
#    with 2-3 sentences telling the LLM how to behave.
#
#    Your instructions should say:
#      - Answer ONLY based on the context (not from memory)
#      - If the context doesn't contain the answer, say "I don't know"
#      - Keep answers concise
#
#    Write in plain English, as if you were talking to a person!
#    (Don't modify anything else — just replace that one line)

RAG_PROMPT_TEMPLATE = """
___WRITE YOUR INSTRUCTIONS HERE___

Context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

In [ ]:
# ── Build the RAG chain (already done — don't modify) ────────────────────
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print('RAG chain ready!')

In [ ]:
# ── Test the RAG chain ────────────────────────────────────────────────────
test_questions = [
    "Who was the first president of the United States?",
    "What is photosynthesis?",
    "What is the best pizza topping?",  # out-of-domain — will it say "I don't know"?
]

for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}\nA: {answer}\n")

### What just happened?

The RAG chain just did the full pipeline for each question:
1. **Retrieved** the 4 most relevant chunks from the vector store
2. **Formatted** them into a "context" string
3. **Filled in** your prompt template with the context and question
4. **Sent** everything to the LLM
5. **Returned** the LLM's answer

### Discussion

1. Did the chain answer the in-domain questions correctly?
2. For the pizza question: Did the LLM say "I don't know" (or something similar)? If it answered about pizza anyway, your prompt needs to be more explicit — go back and improve it!
3. **Experiment**: Try changing your prompt template above and re-running the cells. For example:
   - What happens if you remove the instruction about saying "I don't know"?
   - What happens if you remove the instruction about using only the context?
   - What happens if you add "Answer in one word"?

*Your answer:* [write here]

---
## Task 5 — Evaluation & Comparison

### Why do we need a proper evaluation?

Testing with 3 questions gave us a first impression, but it's not enough to know if our pipeline works *well*. Maybe we got lucky with those specific questions! To properly evaluate, we need to:
1. Test on **many** questions (we'll use 10)
2. Have **ground-truth answers** to compare against
3. Use a **metric** (a number) to measure performance

### Our metric: substring accuracy

We use a simple but effective metric called **substring accuracy**:

> For each question, check if the expected answer appears *somewhere* inside the generated answer (ignoring upper/lower case).

**Examples:**
| Gold answer | Generated answer | Correct? | Why? |
|-------------|-----------------|----------|------|
| "France" | "The answer is France, a country in Europe" | ✓ Yes | "france" is inside the text |
| "France" | "Germany" | ✗ No | "france" is NOT inside the text |
| "1066" | "The Battle of Hastings took place in 1066" | ✓ Yes | "1066" is inside the text |

### RAG vs. Naive: does retrieval actually help?

To understand whether retrieval adds value, we compare two systems:
- **RAG chain**: retrieves relevant chunks, then answers (what we built — the "open-book exam")
- **Naive chain**: sends only the question to the LLM, with NO context at all (the "closed-book exam")

---

### 🎯 What you need to do

Two small tasks:

1. **Build the naive chain** (Task 5a) — The code is almost complete. You just need to connect three components with the `|` operator: `naive_prompt | llm | StrOutputParser()`. This is the same pattern as the RAG chain, but without the retriever.

2. **Compute accuracy** (Task 5b) — The formula is already written. You just need to tell Python which variable contains the answers for each chain: `rag_answers` for the RAG chain, `naive_answers` for the naive chain.

**This is what the completed code should look like:**
```python
# Task 5a:
naive_chain = naive_prompt | llm | StrOutputParser()

# Task 5b:
# ... zip(rag_answers, gold_answers) ...    (for RAG)
# ... zip(naive_answers, gold_answers) ...  (for naive)
```

In [ ]:
# ── Build evaluation set from SQuAD ──────────────────────────────────────
eval_set = helper.build_eval_set(squad, documents, n=30)
print(f"Evaluation set: {len(eval_set)} questions")
print(f"\nExamples:")
for i, ex in enumerate(eval_set[:3]):
    print(f"  [{i+1}] Q: {ex['question']}  →  A: {ex['answer']}")

In [ ]:
# ── Run RAG chain on evaluation questions ────────────────────────────────
eval_questions = [ex["question"] for ex in eval_set[:10]]
gold_answers = [ex["answer"] for ex in eval_set[:10]]

print("Running RAG chain on 10 evaluation questions...\n")
rag_answers = []
for i, q in enumerate(eval_questions):
    answer = rag_chain.invoke(q)
    rag_answers.append(answer)
    correct = gold_answers[i].lower() in answer.lower()
    mark = "✓" if correct else "✗"
    print(f"  [{i+1}/10] {mark}  Q: {q}")
    print(f"           A: {answer}  |  Gold: {gold_answers[i]}\n")

In [ ]:
# ── Task 5a: Build the naive chain (the "closed-book" version) ───────────
#
# The naive chain sends ONLY the question to the LLM, with NO context.
# Three components connected with the | operator:
#   naive_prompt  →  llm  →  StrOutputParser()

naive_prompt = ChatPromptTemplate.from_template(
    "Answer the following question concisely:\n\n{question}"
)

# 🎯 COMPLETE THIS LINE: replace ___ with: naive_prompt | llm | StrOutputParser()
naive_chain = ___

print('Naive chain ready!')

In [ ]:
# ── Run naive chain on the same questions ────────────────────────────────
print("Running naive chain (NO retrieval) on the same 10 questions...\n")
naive_answers = []
for i, q in enumerate(eval_questions):
    answer = naive_chain.invoke(q)
    naive_answers.append(answer)
    correct = gold_answers[i].lower() in answer.lower()
    mark = "✓" if correct else "✗"
    print(f"  [{i+1}/10] {mark}  Q: {q}")
    print(f"           A: {answer}\n")

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────
helper.display_comparison_table(eval_questions, rag_answers, naive_answers, gold_answers)

In [ ]:
# ── Task 5b: Compute substring accuracy ──────────────────────────────────
#
# How this code works:
#   zip(answers, gold)  →  pairs up each generated answer with its gold answer
#   gold.lower() in gen.lower()  →  checks if the gold answer appears in the generated one
#   sum(1 for ... if ...)  →  counts how many are correct
#   / len(eval_questions)  →  divides by total to get a fraction (e.g., 0.7 = 70%)

# 🎯 COMPLETE THIS LINE: replace ___ with: rag_answers
rag_accuracy = sum(
    1 for gen, gold in zip(___, gold_answers)
    if gold.lower() in gen.lower()
) / len(eval_questions)

# 🎯 COMPLETE THIS LINE: replace ___ with: naive_answers
naive_accuracy = sum(
    1 for gen, gold in zip(___, gold_answers)
    if gold.lower() in gen.lower()
) / len(eval_questions)

print(f"RAG accuracy:   {rag_accuracy:.0%} ({int(rag_accuracy * len(eval_questions))}/{len(eval_questions)})")
print(f"Naive accuracy: {naive_accuracy:.0%} ({int(naive_accuracy * len(eval_questions))}/{len(eval_questions)})")

In [ ]:
# ── Visualize the comparison ──────────────────────────────────────────────
helper.plot_evaluation_results({
    "RAG Accuracy": rag_accuracy,
    "Naive Accuracy": naive_accuracy,
})

### Final Discussion

Congratulations — you've built a complete RAG pipeline from scratch! Now reflect on what you've learned:

1. **Which chain won?** Did the RAG chain outperform the naive chain? By how much?

2. **Look at the comparison table:** Are there questions where the naive chain got it right but RAG didn't? Why might that happen? (Hint: think about what happens if the retriever returns irrelevant chunks.)

3. **Types of knowledge:** For which types of questions does RAG help the most?
   - Well-known facts (e.g., "What is the capital of France?") — does the LLM already know these?
   - Specific details from Wikipedia paragraphs (e.g., exact dates, names, numbers) — would the LLM know these without help?

4. **Real-world applications:** Can you think of a scenario where RAG would be *essential*? (Hint: think about information that the LLM has definitely never seen during training — like your company's internal documents, or a new product manual.)

5. **What we built today:**
   - A knowledge base from Wikipedia paragraphs
   - A chunking strategy to split documents into searchable pieces
   - An embedding + vector store system for semantic search
   - A prompt template that controls the LLM's behavior
   - An evaluation framework to measure performance

   Which of these steps do you think had the biggest impact on the final result?

*Your answer:* [write here]